# NB1b v2 - Fine-tune your trained 10-class YOLOv8n on MOCS (without forgetting PPE)
Start point: your Kaggle model (Hardhat, Mask, NO-Hardhat, NO-Mask, NO-Safety Vest, Person, Safety Cone, Safety Vest, machinery, vehicle).
Inputs (both on Drive in `cv5570_risk/data/`): `css_export.zip` (made by the export cell in your Kaggle notebook) and `MOCS_v1i_yolo26.zip`.

**Why not just train on MOCS?** MOCS here only has `Worker` boxes. Training the 10-class model on it alone would teach it that helmets, vests and machinery are *background* and it would forget them. So the training set is a mix:
- **MOCS images** - `Person` boxes come from the MOCS ground truth; the other 9 classes are pseudo-labelled by your old model (only confident ones)
- **CSS replay** - a random sample of your original dataset with its real labels

**Honest evaluation:** before/after on the CSS test split (all 10 classes, to catch regressions) and on the MOCS test split (Person only, the thing we want to improve). MOCS test is never trained on. Runtime: GPU (T4).

In [ ]:
!pip -q install "ultralytics>=8.4.0" scikit-learn joblib pandas pyyaml
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, json, random, shutil, zipfile, gc
from collections import Counter
import numpy as np, pandas as pd, joblib, yaml, torch
from ultralytics import YOLO

ROOT = '/content/drive/MyDrive/cv5570_risk'
for d in ['weights', 'data', 'runs']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

CSS_EXPORT_ZIP = f'{ROOT}/data/css_export.zip'          # from the Kaggle export cell
MOCS_ZIP       = f'{ROOT}/data/MOCS_v1i_yolo26.zip'

N_MOCS      = 600     # MOCS train images used (of 1018)
NEG_FRAC    = 0.10    # share of MOCS sample that has no workers (hard negatives)
N_CSS       = 800     # replay images from your original dataset (of ~2600). More = less forgetting
PSEUDO_CONF = 0.50    # old-model confidence needed to keep a pseudo-label on MOCS images
SEED        = 0
IMGSZ       = 960     # train + new-model eval size (MOCS workers are small). Old model was trained at 640
EPOCHS      = 40
LR0         = 0.0005  # gentle: we are fine-tuning, not training from scratch
FREEZE      = 0       # set 10 to freeze the backbone (even less forgetting, a bit less gain on MOCS)

WORK = '/content/work'
CSS_RAW, MOCS_RAW = '/content/css_export', '/content/mocs_raw'

## Step 1 - Unpack and load the existing model

In [ ]:
if not os.path.isdir(CSS_RAW):  zipfile.ZipFile(CSS_EXPORT_ZIP).extractall(CSS_RAW)
if not os.path.isdir(MOCS_RAW): zipfile.ZipFile(MOCS_ZIP).extractall(MOCS_RAW)

OLD_WEIGHTS = glob.glob(f'{CSS_RAW}/**/old_yolov8n_best.pt', recursive=True)[0]
CSS_DIR = os.path.join(os.path.dirname(OLD_WEIGHTS), 'css-data')
shutil.copy(OLD_WEIGHTS, f'{ROOT}/weights/old_yolov8n_best.pt')

old = YOLO(OLD_WEIGHTS)
NAMES = [old.names[i] for i in range(len(old.names))]
PERSON_ID = NAMES.index('Person')
print(NAMES, '| Person id =', PERSON_ID)
print({s: len(glob.glob(f'{CSS_DIR}/{s}/images/*.*')) for s in ['train', 'valid', 'test']}, 'CSS images')

## Step 2 - Build the mixed dataset

In [ ]:
def read_boxes(lbl_path):
    """YOLO label file -> [(cls, x, y, w, h)]; polygons (MOCS) are converted to their bounding box."""
    out = []
    if os.path.exists(lbl_path):
        for line in open(lbl_path):
            p = line.split()
            if len(p) < 5: continue
            v = list(map(float, p[1:]))
            if len(v) == 4: x, y, w, h = v
            else:
                xs, ys = v[0::2], v[1::2]
                x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
                x, y, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
            if w > 0.004 and h > 0.004: out.append((int(p[0]), x, y, w, h))
    return out

lbl_of = lambda img: img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'

def put(split_dir, img, rows, prefix=''):
    os.makedirs(f'{split_dir}/images', exist_ok=True); os.makedirs(f'{split_dir}/labels', exist_ok=True)
    name = prefix + os.path.basename(img)
    shutil.copy(img, f'{split_dir}/images/{name}')
    with open(f'{split_dir}/labels/{os.path.splitext(name)[0]}.txt', 'w') as f:
        for c, x, y, w, h in rows: f.write(f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n')

def predict_chunks(model, paths, imgsz, conf, n=8):
    # NB: ultralytics treats a list of paths as ONE batch -> always feed small chunks
    for k in range(0, len(paths), n):
        yield from model.predict(paths[k:k + n], imgsz=imgsz, conf=conf, verbose=False)

def mocs_rows(img, pseudo):
    gt = [(PERSON_ID, x, y, w, h) for _, x, y, w, h in read_boxes(lbl_of(img))]
    return gt + pseudo.get(img, [])

def pseudo_label(imgs):
    """old model's confident NON-person detections on MOCS images (person comes from the MOCS ground truth)"""
    out = {}
    for res in predict_chunks(old, imgs, 640, PSEUDO_CONF):
        rows = []
        for c, xywhn in zip(res.boxes.cls.tolist(), res.boxes.xywhn.tolist()):
            if int(c) != PERSON_ID: rows.append((int(c), *xywhn))
        out[res.path] = rows
    return out

shutil.rmtree(WORK, ignore_errors=True)
MIX, EV = f'{WORK}/mix', f'{WORK}/mocs_eval'
rng = random.Random(SEED)

# --- MOCS train sample (+ pseudo labels) ---
mocs_train = sorted(glob.glob(f'{MOCS_RAW}/train/images/*.*'))
pos = [p for p in mocs_train if read_boxes(lbl_of(p))]; posset = set(pos)
neg = [p for p in mocs_train if p not in posset]
n_neg = min(len(neg), int(N_MOCS * NEG_FRAC)); n_pos = min(len(pos), N_MOCS - n_neg)
mocs_s = rng.sample(pos, n_pos) + rng.sample(neg, n_neg)
mocs_val = sorted(glob.glob(f'{MOCS_RAW}/valid/images/*.*'))
mocs_test = sorted(glob.glob(f'{MOCS_RAW}/test/images/*.*'))
pl = pseudo_label(mocs_s + mocs_val)
for img in mocs_s:   put(f'{MIX}/train', img, mocs_rows(img, pl), 'mocs_')
for img in mocs_val: put(f'{MIX}/val',   img, mocs_rows(img, pl), 'mocs_')

# --- CSS replay (original labels) ---
css_train = sorted(glob.glob(f'{CSS_DIR}/train/images/*.*'))
for img in rng.sample(css_train, min(N_CSS, len(css_train))): put(f'{MIX}/train', img, read_boxes(lbl_of(img)), 'css_')
for img in sorted(glob.glob(f'{CSS_DIR}/valid/images/*.*')):  put(f'{MIX}/val',   img, read_boxes(lbl_of(img)), 'css_')

# --- MOCS eval copies: boxes only, single class 'Worker' (id 0). val = thresholds/calibration, test = reported numbers ---
for img in mocs_val:  put(f'{EV}/val',  img, [(0, x, y, w, h) for _, x, y, w, h in read_boxes(lbl_of(img))])
for img in mocs_test: put(f'{EV}/test', img, [(0, x, y, w, h) for _, x, y, w, h in read_boxes(lbl_of(img))])

MIX_YAML = f'{WORK}/mixed.yaml'
yaml.safe_dump({'path': MIX, 'train': 'train/images', 'val': 'val/images', 'nc': len(NAMES),
                'names': {i: n for i, n in enumerate(NAMES)}}, open(MIX_YAML, 'w'))

cnt = Counter()
for f in glob.glob(f'{MIX}/train/labels/*.txt'):
    for line in open(f): cnt[NAMES[int(line.split()[0])]] += 1
print('train images:', len(glob.glob(f'{MIX}/train/images/*')), '| val images:', len(glob.glob(f'{MIX}/val/images/*')))
print('train boxes per class:', dict(cnt))

## Step 3 - Evaluator and BASELINE of the existing model
Same matcher for old and new (IoU 0.5). The old model is scored at 640 (how it was trained) *and* at the new size, so the comparison is fair.

In [ ]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0

def load_gt(img_path, shape):
    H, W = shape[:2]
    return [(c, [(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H]) for c, x, y, w, h in read_boxes(lbl_of(img_path))]

def collect(weights, img_dir, imgsz, cls_map, names, conf=0.05):
    """cls_map: model class id -> dataset class id. Returns samples[name]=(confs, is_tp) and n_gt[name]."""
    m = YOLO(weights) if isinstance(weights, str) else weights
    paths = sorted(glob.glob(img_dir + '/*.*'))
    samples = {n: ([], []) for n in names}; n_gt = {n: 0 for n in names}
    for res in predict_chunks(m, paths, imgsz, conf):
        gt = load_gt(res.path, res.orig_shape)
        for c, _ in gt: n_gt[names[c]] += 1
        used = set()
        for i in sorted(range(len(res.boxes)), key=lambda i: -float(res.boxes.conf[i])):
            mc = int(res.boxes.cls[i])
            if mc not in cls_map: continue
            c = cls_map[mc]; cf = float(res.boxes.conf[i]); bx = res.boxes.xyxy[i].tolist()
            best, bj = 0, None
            for j, (gc, gb) in enumerate(gt):
                if gc == c and j not in used:
                    v = iou(bx, gb)
                    if v > best: best, bj = v, j
            ok = best >= 0.5
            if ok: used.add(bj)
            samples[names[c]][0].append(cf); samples[names[c]][1].append(int(ok))
    return samples, n_gt

def ap50(cf, tp, n_gt):
    if n_gt == 0 or not cf: return float('nan')
    o = np.argsort(-np.array(cf)); tp = np.array(tp)[o]
    ctp = np.cumsum(tp); cfp = np.cumsum(1 - tp)
    rec = ctp / n_gt; prec = ctp / (ctp + cfp)
    rec = np.concatenate([[0], rec, [1]]); prec = np.concatenate([[1], prec, [0]])
    prec = np.maximum.accumulate(prec[::-1])[::-1]
    idx = np.where(rec[1:] != rec[:-1])[0]
    return float(np.sum((rec[idx + 1] - rec[idx]) * prec[idx + 1]))

def prf(cf, tp, n_gt, thr):
    cf, tp = np.array(cf), np.array(tp)
    k = cf >= thr; ntp = tp[k].sum(); nd = k.sum()
    P = ntp / nd if nd else 0.0; R = ntp / n_gt if n_gt else 0.0
    return P, R, (2 * P * R / (P + R) if P + R else 0.0)

def best_f1_thr(cf, tp, n_gt):
    grid = np.arange(0.05, 0.95, 0.01)
    f = np.convolve([prf(cf, tp, n_gt, t)[2] for t in grid], np.ones(5) / 5, mode='same')
    return float(np.clip(grid[int(np.argmax(f))], 0.1, 0.7))

def summarize(weights, tag, sizes):
    """-> css (10-class AP50 / recall @0.25, per size) and mocs (Person only) DataFrames"""
    m = YOLO(weights); css, mocs = {}, {}
    for s in sizes:
        sm, ng = collect(m, f'{CSS_DIR}/test/images', s, {i: i for i in range(len(NAMES))}, NAMES)
        for n in NAMES:
            css[(n, f'{tag}@{s}')] = dict(AP50=ap50(*sm[n], ng[n]), R=prf(*sm[n], ng[n], 0.25)[1], n_gt=ng[n])
        sm, ng = collect(m, f'{EV}/test/images', s, {PERSON_ID: 0}, ['Worker'])
        P, R, F = prf(*sm['Worker'], ng['Worker'], 0.25)
        mocs[f'{tag}@{s}'] = dict(n_gt=ng['Worker'], P=P, R=R, F1=F, AP50=ap50(*sm['Worker'], ng['Worker']))
    css = pd.DataFrame({t: {n: css[(n, t)]['AP50'] for n in NAMES} for t in sorted({k[1] for k in css})})
    return css, pd.DataFrame(mocs).T

old_css, old_mocs = summarize(OLD_WEIGHTS, 'old', [640, IMGSZ])
old_css.to_csv(f'{ROOT}/weights/baseline_old_css_ap50.csv'); old_mocs.to_csv(f'{ROOT}/weights/baseline_old_mocs_person.csv')
print('CSS test AP50 per class (old model):'); display(old_css.round(3))
print('MOCS test, Person only (old model):');  display(old_mocs.round(3))

## Step 4 - Fine-tune the existing model (checkpoints go to Drive)

In [ ]:
model = YOLO(OLD_WEIGHTS)                          # 10-class head is kept, so all PPE knowledge is the starting point
args = dict(data=MIX_YAML, imgsz=IMGSZ, epochs=EPOCHS, patience=10, batch=16,     # batch=8 if out-of-memory
            optimizer='AdamW', lr0=LR0, warmup_epochs=1, close_mosaic=5,
            workers=2, cache=False, amp=True,
            project=f'{ROOT}/runs', name='ppe_worker_ft', exist_ok=True)
if FREEZE: args['freeze'] = FREEZE
model.train(**args)

In [ ]:
# ONLY if Colab disconnected mid-training (re-run Steps 1-2 first, then this):
# YOLO(f'{ROOT}/runs/ppe_worker_ft/weights/last.pt').train(resume=True)

## Step 5 - Before vs after, thresholds, calibration, save
Look at the **CSS table first**: if any PPE class dropped a lot, increase `N_CSS` or set `FREEZE = 10` and re-train.

In [ ]:
try: del model
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

NEW_BEST = f'{ROOT}/runs/ppe_worker_ft/weights/best.pt'
new_css, new_mocs = summarize(NEW_BEST, 'new', [640, IMGSZ])

css_cmp = old_css.join(new_css)
css_cmp['delta_best'] = new_css.max(axis=1) - old_css.max(axis=1)      # best size of each model
css_cmp.to_csv(f'{ROOT}/weights/before_after_css.csv')
print('CSS test AP50 (regression check) - delta_best < -0.05 means that class got worse:'); display(css_cmp.round(3))
print('classes that regressed:', list(css_cmp.index[css_cmp['delta_best'] < -0.05]) or 'none')

mocs_cmp = pd.concat([old_mocs, new_mocs]); mocs_cmp.to_csv(f'{ROOT}/weights/before_after_mocs_person.csv')
print('MOCS test, Person only (what we wanted to improve):'); display(mocs_cmp.round(3))

In [ ]:
# thresholds + isotonic calibration, fitted on val (never on test).
#   9 PPE/other classes: CSS valid   |   Person: MOCS valid (the harder, small-worker domain)
ID = {i: i for i in range(len(NAMES))}
s_css, n_css = collect(NEW_BEST, f'{CSS_DIR}/valid/images', IMGSZ, ID, NAMES)
s_moc, n_moc = collect(NEW_BEST, f'{EV}/val/images', IMGSZ, {PERSON_ID: 0}, ['Worker'])

samples = {n: (s_css[n], n_css[n]) for n in NAMES if n != 'Person'}
samples['Person'] = (s_moc['Worker'], n_moc['Worker'])
thr = {n: best_f1_thr(*s[0], s[1]) for n, s in samples.items() if s[1] > 0}

from sklearn.isotonic import IsotonicRegression
calib = {}
for n, ((cf, ok), ng) in samples.items():
    if len(cf) >= 20 and 0 < sum(ok) < len(ok):
        calib[n] = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(cf, ok)

shutil.copy(NEW_BEST, f'{ROOT}/weights/det_ppe_worker_best.pt')
json.dump(thr, open(f'{ROOT}/weights/class_thresholds_ppe_worker.json', 'w'), indent=1)
joblib.dump(calib, f'{ROOT}/weights/calib_ppe_worker.joblib')
print('thresholds:', {k: round(v, 2) for k, v in thr.items()})
print('calibrated classes:', list(calib))
print('saved to', f'{ROOT}/weights/', '-> det_ppe_worker_best.pt, class_thresholds_ppe_worker.json, calib_ppe_worker.joblib')

## Step 6 - Look at it (a MOCS test image and a CSS test image per row)

In [ ]:
from IPython.display import display, Image as IPImage
m = YOLO(f'{ROOT}/weights/det_ppe_worker_best.pt')
pick = random.sample(sorted(glob.glob(f'{EV}/test/images/*.*')), 3) + random.sample(sorted(glob.glob(f'{CSS_DIR}/test/images/*.*')), 3)
for i, p in enumerate(pick):
    r = m.predict(p, imgsz=IMGSZ, conf=0.3, verbose=False)[0]
    r.save(f'/content/_vis{i}.jpg'); display(IPImage(f'/content/_vis{i}.jpg', width=640))

In [ ]:
# ===== Step 7: collect the important outputs from Drive and download them =====
from google.colab import drive, files
import os, shutil, json
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/cv5570_risk'
W, RUN = f'{ROOT}/weights', f'{ROOT}/runs/ppe_worker_ft'
OUT = '/content/ppe_worker_outputs'
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)

wanted = {
    f'{W}/det_ppe_worker_best.pt':            'det_ppe_worker_best.pt',          # the model to use
    f'{W}/class_thresholds_ppe_worker.json':  'class_thresholds_ppe_worker.json',
    f'{W}/calib_ppe_worker.joblib':           'calib_ppe_worker.joblib',
    f'{W}/before_after_css.csv':              'tables/before_after_css.csv',
    f'{W}/before_after_mocs_person.csv':      'tables/before_after_mocs_person.csv',
    f'{W}/baseline_old_css_ap50.csv':         'tables/baseline_old_css_ap50.csv',
    f'{W}/baseline_old_mocs_person.csv':      'tables/baseline_old_mocs_person.csv',
    f'{RUN}/weights/best.pt':                 'training/best.pt',                # backup copy of the model
    f'{RUN}/weights/last.pt':                 'training/last.pt',                # only needed to resume training
    f'{RUN}/results.csv':                     'training/results.csv',
    f'{RUN}/results.png':                     'training/results.png',
    f'{RUN}/args.yaml':                       'training/args.yaml',
}
copied, missing = [], []
for src, dst in wanted.items():
    if os.path.exists(src):
        os.makedirs(os.path.dirname(f'{OUT}/{dst}') or OUT, exist_ok=True)
        shutil.copy(src, f'{OUT}/{dst}'); copied.append(dst)
    else:
        missing.append(src)

json.dump(['Hardhat','Mask','NO-Hardhat','NO-Mask','NO-Safety Vest','Person','Safety Cone','Safety Vest','machinery','vehicle'],
          open(f'{OUT}/classes.json', 'w'))                                        # class id order of the model
print('copied :', copied)
print('MISSING:', missing or 'none')

shutil.make_archive('/content/ppe_worker_outputs', 'zip', OUT)
print('zip size (MB):', round(os.path.getsize('/content/ppe_worker_outputs.zip') / 1e6, 1))
files.download('/content/ppe_worker_outputs.zip')